In [2]:
! uv pip install qiskit qiskit-aer matplotlib


Audited 3 packages in 26ms


In [21]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import time
import random

In [27]:
def classical_search(target, data):
    """Busca clássica (força bruta): percorre todos os itens até achar o alvo"""
    for i, value in enumerate(data):
        if value == target:
            return i  # retorna o índice do alvo
    return -1  

# Cria uma lista desordenada de N elementos
N = 1024  # aqui você pode ajustar o tamanho da lista
data = [random.randint(0, 9999) for _ in range(N)]

# Escolhe um valor aleatório da lista como alvo
target = data[random.randint(0, N - 1)]

# Executa a busca clássica
start = time.perf_counter()
index_found = classical_search(target, data)
elapsed_time = time.perf_counter() - start

# Visualização dos resultados
print(f"\n Valor alvo: {target}")
print(f" Posição encontrada: {index_found}")
print(f" Tempo de execução: {elapsed_time:.6f} segundos")
print(f" Itens verificados: {N}")



 Valor alvo: 5343
 Posição encontrada: 44
 Tempo de execução: 0.000052 segundos
 Itens verificados: 1024


In [26]:
# Função para implementar o algoritmo de Grover
def grover_search(num_qubits, target_state):
    """Simula o algoritmo de Grover para buscar 'target_state' em uma lista de 2^n elementos.
        Exemplo: grover_search(3, "101")
    """

    if len(target_state) != num_qubits:
        raise ValueError("O comprimento de 'target_state' deve ser igual a 'num_qubits'.")
    
    # Mede tempo total de execução
    start_time = time.time()
    
    # registradores quânticos e clássicos
    qreg = QuantumRegister(num_qubits, "q")
    creg = ClassicalRegister(num_qubits, "c")
    qc = QuantumCircuit(qreg, creg)

    # inicializar em superposição
    qc.h(qreg)

    # ORÁCULO - marcar o estado alvo
    # Inverte os qubits que são 0 no estado alvo (para alinhar com CZ)
    for i, bit in enumerate(target_state):
        if bit == '0':
            qc.x(qreg[i])
    
    # Aplica CZ para marcar o estado alvo
    if num_qubits == 1:
        qc.z(qreg[0])
    else:
        qc.h(qreg[-1])
        qc.mcx(qreg[:-1], qreg[-1])  # multi-controlled X (Toffoli generalizado)
        qc.h(qreg[-1])
    
    # Desfaz as inversões
    for i, bit in enumerate(target_state):
        if bit == '0':
            qc.x(qreg[i])

    # DIFUSOR
    qc.h(qreg)
    qc.x(qreg)
    qc.h(qreg[-1])
    qc.cx(qreg[:-1], qreg[-1])
    qc.h(qreg[-1])
    qc.x(qreg)
    qc.h(qreg)

    # Medição
    qc.measure(qreg, creg)

    # Simulador
    sim = AerSimulator()
    job = sim.run(qc, shots=1024)
    result = job.result()
    counts = result.get_counts()

     # Tempo TOtal
    elapsed_time = time.perf_counter() - start_time

    # Identifica o estado mais medido (resultado mais provável)
    found_state = max(counts, key=counts.get)

    # Visualização dos resultados
    print("Resultados da medição:", counts)
    print("\n Valor alvo:", target_state)
    print(" Estado encontrado:", found_state)
    print(" Tempo de execução: {:.6f} segundos".format(elapsed_time))
    print(" Itens verificados (equivalentes):", 2 ** num_qubits)
    plot_histogram(counts)
    plt.show()

# Exemplo de uso
grover_search(num_qubits=3, target_state="101")


Resultados da medição: {'001': 495, '011': 126, '000': 144, '100': 134, '111': 125}

 Valor alvo: 101
 Estado encontrado: 001
 Tempo de execução: -1760596575.408355 segundos
 Itens verificados (equivalentes): 8
